# Hand joints from IR stereo: training baseline (Kaggle)

Fetches the data straight from the shared Google Drive (no login, only the files needed), trains, and reports the
3D joint error in mm. Two data sources, picked by `DATA`:

| `DATA` | Drive folder | input | target |
|---|---|---|---|
| `two_cam` | `sessions/` (this repo: both 3Dis, `frames.bin`, `groups.json`, `mano/smoothed.npz`) | 4 IR images | 21 MANO joints, mm, cam_A frame |
| `single_cam` | `dataset/` (older capture: one 3Di, 10 FPS, PNG pairs, `leap/*.json`) | 2 IR images | 21 LeapC joints, mm |

`auto` uses `two_cam` when any complete two-camera session is on the Drive, otherwise `single_cam`. On 2026-11-11 the
two-camera sessions on the Drive were empty folders, so `auto` gives `single_cam` (~2,600 frames).

**The model is a placeholder (ResNet-18 regressing 21 joints).** Swap `make()` for the real one.

### Before running
1. Settings: Accelerator **GPU T4 x2**, **Internet on** (needs a phone-verified Kaggle account).
2. Only if `USE_MANO = True`: Add-ons, Secrets: `MANO_USER`, `MANO_PASS` (your mano.is.tue.mpg.de login), attached here.
3. Run all, or Save Version, Save & Run All to run in the background. `best.pt` lands in the output.

### How long (single_cam, ~2,600 frames)
Listing the Drive ~4 min, downloading ~7,800 small files ~10-15 min, training 30 epochs ~3-8 min: about 20-30 min
(measured locally: listing 3.9 min; 120 labels 0.2 min and 240 images 0.4 min with 16 parallel downloads).
The download prints its time left as it goes, the **Time estimate** cell prints the measured training time before
training starts, and every epoch prints the time left.

In [ ]:
DRIVE_URL = "https://drive.google.com/drive/folders/19yETUSzehXSXpiADBFPc1n9XN8-x41jp"
DATA = "auto"      # "auto", "two_cam" or "single_cam" (see the table above)
EPOCHS = 30
IMG = 256          # images are resized from 384 x 384 to IMG x IMG
BATCH = 64
LR = 3e-4
MIN_CONF = 0.0     # single_cam: drop frames whose LeapC confidence is below this
VAL_FRAC = 0.2     # two_cam: the last 20 % (in time) of every session is validation
MAX_FRAMES = 0     # > 0: use only this many frames (quick test run)
USE_MANO = False   # download + convert MANO (only needed once the model outputs MANO parameters)
WORK = "/tmp/data"                                                     # downloads (/kaggle/working is capped at 20 GB)
OUT = "/kaggle/working" if __import__("os").path.isdir("/kaggle/working") else "."  # best.pt

## Google Drive access

In [ ]:
import collections, csv, glob, html, io, json, math, os, re, shutil, subprocess, sys, time, zipfile
from concurrent.futures import ThreadPoolExecutor
import requests

T_START = time.time()
took = lambda t: f"{(time.time() - t) / 60:.1f} min"
S = requests.Session()
DL = "https://drive.usercontent.google.com/download"

def list_folder(fid):
    # Drive's embedded folder view lists every entry without a login (gdown stops at 50 per folder)
    for attempt in range(5):
        try:
            t = S.get(f"https://drive.google.com/embeddedfolderview?id={fid}", timeout=60).text
            break
        except requests.RequestException:
            time.sleep(2 ** attempt)
    out = []
    for chunk in t.split('class="flip-entry"')[1:]:
        eid = re.search(r'id="entry-([\w-]+)"', chunk).group(1)
        name = html.unescape(re.search(r'flip-entry-title">([^<]*)', chunk).group(1))
        out.append((name, eid, "/drive/folders/" in chunk))
    return out

def walk(fid, path):
    items = {}
    for name, eid, folder in list_folder(fid):
        p = f"{path}/{name}"
        if folder:
            items.update(walk(eid, p))
        else:
            items[p] = eid
    return items

def download(fid, dst):
    if os.path.exists(dst):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(6):
        try:
            r = S.get(DL, params={"id": fid, "export": "download"}, stream=True, timeout=120)
            if "text/html" in r.headers.get("Content-Type", ""):  # big file: confirm on Drive's "can't scan" page
                form = dict(re.findall(r'name="(\w+)" value="([^"]*)"', r.text))
                if "confirm" not in form:
                    raise IOError(re.sub(r"<[^>]+>|\s+", " ", r.text)[:200])
                r = S.get(DL, params=form, stream=True, timeout=600)
                if "text/html" in r.headers.get("Content-Type", ""):
                    raise IOError("Drive refused the download (quota?)")
            r.raise_for_status()
            with open(dst + ".part", "wb") as f:
                for b in r.iter_content(1 << 20):
                    f.write(b)
            os.replace(dst + ".part", dst)
            return
        except Exception as e:
            err = e
            time.sleep(2 ** attempt)
    raise IOError(f"{dst}: {err}")

def download_many(pairs, what):
    # pairs: [(drive id, local path)]; prints progress and time left
    pairs = [p for p in pairs if not os.path.exists(p[1])]
    t, n = time.time(), len(pairs)
    with ThreadPoolExecutor(16) as ex:
        for k, _ in enumerate(ex.map(lambda p: download(*p), pairs), 1):
            if k % max(1, n // 10) == 0 or k == n:
                el = time.time() - t
                print(f"  {what}: {k}/{n} files, {el / 60:.1f} min, ~{el / k * (n - k) / 60:.1f} min left", flush=True)

t = time.time()
ROOT = re.search(r"folders/([\w-]+)", DRIVE_URL).group(1)
top = {name: eid for name, eid, _ in list_folder(ROOT)}
print("Drive folders:", list(top))
index = {}
for name in ("sessions", "dataset"):
    if name in top:
        index.update(walk(top[name], name))
print(f"{len(index)} files listed in {took(t)}")

## Pick the data

In [ ]:
NEED = ["session.json", "groups.json", "cam_A/frames.bin", "cam_A/frames.csv", "cam_B/frames.bin", "cam_B/frames.csv",
        "mano/smoothed.npz"]
two = sorted(s for s in {p.split("/")[1] for p in index if p.startswith("sessions/") and p.count("/") >= 2}
             if all(f"sessions/{s}/{n}" in index for n in NEED))
zips = sorted(p for p in index if re.fullmatch(r"sessions/[^/]+\.zip", p))

frames = collections.defaultdict(dict)  # single_cam: (session, frame) -> {"left": id, "right": id, "leap": id}
for p, fid in index.items():
    m = re.fullmatch(r"dataset/(session_[^/]+)/(left|right|leap)/(frame_\d+)\.(png|json)", p)
    if m:
        frames[(m[1], m[3])][m[2]] = fid
single = sorted(k for k, v in frames.items() if len(v) == 3)

mode = DATA if DATA != "auto" else ("two_cam" if two or zips else "single_cam")
print(f"two_cam: {len(two)} complete sessions + {len(zips)} zips;  single_cam: {len(single)} complete frames")
print(f"-> using {mode}")

## Download

In [ ]:
t = time.time()
if mode == "two_cam":
    download_many([(index[f"sessions/{s}/{n}"], f"{WORK}/sessions/{s}/{n}") for s in two for n in NEED], "sessions")
    for z in zips:
        download(index[z], f"{WORK}/{z}")
        shutil.unpack_archive(f"{WORK}/{z}", f"{WORK}/sessions")
    sessions = sorted({os.path.dirname(p) for p in glob.glob(f"{WORK}/sessions/**/groups.json", recursive=True)
                       if all(os.path.exists(f"{os.path.dirname(p)}/{n}") for n in NEED)})
    print(f"{len(sessions)} complete sessions:", [os.path.basename(s) for s in sessions])
else:
    txt = {n: f"{WORK}/dataset/{n}" for n in ("val_sessions.txt", "excluded_frames.txt") if f"dataset/{n}" in index}
    for n, dst in txt.items():
        download(index[f"dataset/{n}"], dst)
    val_sessions = set(open(txt["val_sessions.txt"]).read().split()) if "val_sessions.txt" in txt else set()
    excl = set(open(txt["excluded_frames.txt"]).read().split()) if "excluded_frames.txt" in txt else set()
    keep = [k for k in single if k[0] not in excl and f"{k[0]}/{k[1]}" not in excl]
    print(f"{len(single) - len(keep)} frames excluded (excluded_frames.txt)")
    if MAX_FRAMES:
        keep = keep[:: max(1, len(keep) // MAX_FRAMES)][:MAX_FRAMES]
    loc = lambda k, part, ext: f"{WORK}/dataset/{k[0]}/{part}/{k[1]}.{ext}"
    download_many([(frames[k]["leap"], loc(k, "leap", "json")) for k in keep], "labels")
    conf = {k: json.load(open(loc(k, "leap", "json"))).get("confidence", 1.0) for k in keep}
    print("LeapC confidence p10 / p50 / p90:", [round(float(q), 3) for q in __import__("numpy").percentile(list(conf.values()), [10, 50, 90])])
    keep = [k for k in keep if conf[k] >= MIN_CONF]
    download_many([(frames[k][p], loc(k, p, "png")) for k in keep for p in ("left", "right")], "images")
print(f"download {took(t)}")

## Load into memory

In [ ]:
import cv2
import numpy as np

t = time.time()
X, Y, S_, V = [], [], [], []  # images (C, IMG, IMG) uint8, joints (21, 3) mm, session name, is validation
small = lambda im: cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)
if mode == "two_cam":
    for s in sessions:
        meta = json.load(open(f"{s}/session.json"))
        w, h = meta["image_size"]
        z = np.load(f"{s}/mano/smoothed.npz")
        a2b = {g["frames"]["cam_A"]: g["frames"]["cam_B"] for g in json.load(open(f"{s}/groups.json"))["groups"]}
        off = {c: {int(r["frame_id"]): int(r["offset"]) for r in csv.DictReader(open(f"{s}/{c}/frames.csv"))}
               for c in ("cam_A", "cam_B")}
        mm = {c: np.memmap(f"{s}/{c}/frames.bin", np.uint8, "r") for c in ("cam_A", "cam_B")}
        pair = lambda c, f: mm[c][off[c][f]:off[c][f] + 2 * w * h].reshape(2, h, w)
        n0 = len(X)
        for i, fa in enumerate(z["frames"].astype(int)):
            fb = a2b.get(fa)
            if fb is None or fa not in off["cam_A"] or fb not in off["cam_B"]:
                continue
            X.append(np.stack([small(im) for im in (*pair("cam_A", fa), *pair("cam_B", fb))]))
            Y.append(z["joints"][i]); S_.append(os.path.basename(s))
        n = len(X) - n0
        V += [False] * (n - int(n * VAL_FRAC)) + [True] * int(n * VAL_FRAC)  # time split: neighbours are near-duplicates
        if MAX_FRAMES and len(X) >= MAX_FRAMES:
            break
    JOINTS = ["wrist", "index1", "index2", "index3", "middle1", "middle2", "middle3", "pinky1", "pinky2", "pinky3",
              "ring1", "ring2", "ring3", "thumb1", "thumb2", "thumb3", "thumb_tip", "index_tip", "middle_tip",
              "ring_tip", "pinky_tip"]  # MANO order (fit_mano.py)
else:
    for k in keep:
        ims = [cv2.imread(loc(k, p, "png"), cv2.IMREAD_GRAYSCALE) for p in ("left", "right")]
        if any(im is None for im in ims):
            print("unreadable image, skipped:", k)
            continue
        lab = json.load(open(loc(k, "leap", "json")))["joints"]
        names = sorted(lab, key=lambda n: int(n.split("_")[0]))
        X.append(np.stack([small(im) for im in ims]))
        Y.append([[lab[n]["x"], lab[n]["y"], lab[n]["z"]] for n in names]); S_.append(k[0])
    JOINTS = names
    V = [s in val_sessions for s in S_]
    if not any(V) or all(V):  # val sessions missing: fall back to the last 20 % of each session
        V = []
        for s in dict.fromkeys(S_):
            n = S_.count(s)
            V += [False] * (n - int(n * 0.2)) + [True] * int(n * 0.2)
X, Y, S_, V = np.stack(X), np.asarray(Y, np.float32), np.array(S_), np.array(V)
tr, va = np.where(~V)[0], np.where(V)[0]
print(f"{len(X)} frames, {X.shape[1]} images each ({X.nbytes / 1e9:.2f} GB), {len(set(S_))} sessions; "
      f"train {len(tr)}, val {len(va)} ({len(set(S_[va]))} sessions); {took(t)}")

## MANO (only if `USE_MANO`)

In [ ]:
if USE_MANO:
    from kaggle_secrets import UserSecretsClient
    REPO = "/kaggle/working/repo"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/amitabh-anandpd/multi-3di-synced-ir-capture", REPO], check=True)
    sec = UserSecretsClient()
    r = requests.post("https://download.is.tue.mpg.de/download.php?domain=mano&resume=1&sfile=mano_v1_2.zip",
                      data={"username": sec.get_secret("MANO_USER"), "password": sec.get_secret("MANO_PASS")}, timeout=900)
    assert r.content[:2] == b"PK", f"MANO download failed (login / licence?): {r.text[:300]}"
    os.makedirs(f"{REPO}/mano", exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for side in ("LEFT", "RIGHT"):
            name = next(n for n in z.namelist() if n.endswith(f"models/MANO_{side}.pkl"))
            open(f"{REPO}/mano/MANO_{side}.pkl", "wb").write(z.read(name))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "smplx"], check=True)
    subprocess.run([sys.executable, f"{REPO}/tools/convert_mano.py"], check=True, capture_output=True)
    sys.path.insert(0, REPO)
    print("MANO ready in", f"{REPO}/mano (use: from fit_mano import Hand)")
else:
    print("MANO skipped (USE_MANO = False)")

## Model (placeholder: swap `make()` for the real one)

In [ ]:
import torch, torch.nn as nn, torchvision

dev = "cuda" if torch.cuda.is_available() else "cpu"
Xg = torch.from_numpy(X).to(dev)                  # the whole set sits on the GPU
Yg = torch.from_numpy(Y).to(dev)                  # (n, 21, 3) mm
mu, sd = Yg[tr].mean(0), Yg[tr].std(0).mean() + 1e-6
C = X.shape[1]

def prep(x, train=False):
    x = x.float() / 255
    if train:
        x = x * torch.empty(len(x), 1, 1, 1, device=dev).uniform_(0.8, 1.2)  # brightness jitter
    return (x - 0.3) / 0.25

def make():
    net = torchvision.models.resnet18(weights="DEFAULT")
    w = net.conv1.weight.data.mean(1, keepdim=True).repeat(1, C, 1, 1)
    net.conv1 = nn.Conv2d(C, 64, 7, 2, 3, bias=False); net.conv1.weight.data = w
    net.fc = nn.Linear(512, 63)
    net = net.to(dev)
    if torch.cuda.device_count() > 1:
        net = nn.DataParallel(net)
    opt = torch.optim.AdamW(net.parameters(), LR, weight_decay=1e-4)
    return net, opt, torch.cuda.amp.GradScaler(enabled=dev == "cuda")

def predict(net, x, train=False):
    with torch.autocast("cuda", torch.float16, enabled=dev == "cuda"):
        out = net(prep(x, train))
    return out.float().view(-1, 21, 3) * sd + mu    # mm

def step(net, opt, scaler, i):
    loss = (predict(net, Xg[i], True) - Yg[i]).norm(dim=-1).mean() / 10  # mean joint error, cm
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return loss.item() * 10

@torch.no_grad()
def evaluate(net, idx):  # per-frame mean joint error (mm): absolute, and with the wrist at the origin
    net.eval()
    P = torch.cat([predict(net, Xg[idx[b:b + 256]]) for b in range(0, len(idx), 256)])
    net.train()
    T = Yg[idx]
    rel = lambda a: a - a[:, :1]
    return (P - T).norm(dim=-1).mean(1).cpu().numpy(), (rel(P) - rel(T)).norm(dim=-1).mean(1).cpu().numpy()

base = (mu - Yg[va]).norm(dim=-1).mean().item()
print(f"{dev}, {torch.cuda.device_count()} GPU(s); floor to beat (always predict the mean hand): {base:.1f} mm")

## Time estimate

In [ ]:
net, opt, scaler = make()
spe = math.ceil(len(tr) / BATCH)
for k in range(25):
    if k == 5:
        torch.cuda.synchronize() if dev == "cuda" else None; t = time.time()
    step(net, opt, scaler, torch.from_numpy(np.random.choice(tr, BATCH)).to(dev))
torch.cuda.synchronize() if dev == "cuda" else None
per_step = (time.time() - t) / 20
t = time.time(); evaluate(net, va); per_eval = time.time() - t
est = EPOCHS * (spe * per_step + per_eval)
print(f"{per_step * 1000:.0f} ms/step, {spe} steps/epoch, eval {per_eval:.1f} s")
print(f"ESTIMATED TRAINING TIME: {est / 60:.1f} min for {EPOCHS} epochs (elapsed so far {took(T_START)})")
net, opt, scaler = make()  # train from fresh weights

## Train

In [ ]:
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=EPOCHS * spe)
best, t = 1e9, time.time()
for ep in range(EPOCHS):
    perm = torch.from_numpy(np.random.permutation(tr)).to(dev)
    losses = []
    for b in range(spe):
        losses.append(step(net, opt, scaler, perm[b * BATCH:(b + 1) * BATCH]))
        sched.step()
    e, er = evaluate(net, va)
    star = e.mean() < best
    if star:
        best = e.mean()
        torch.save({"model": (net.module if hasattr(net, "module") else net).state_dict(), "mu": mu.cpu(),
                    "sd": sd.cpu(), "img": IMG, "channels": C, "data": mode, "joints": JOINTS}, f"{OUT}/best.pt")
    left = (time.time() - t) / (ep + 1) * (EPOCHS - ep - 1)
    print(f"epoch {ep + 1:3d}/{EPOCHS}  train {np.mean(losses):6.1f} mm  val {e.mean():6.1f} mm  "
          f"wrist-relative {er.mean():5.1f} mm {'*' if star else ' '}  ~{left / 60:.1f} min left", flush=True)
print(f"training {took(t)}, best val {best:.1f} mm, whole run {took(T_START)}")

## Result per session and per joint (best checkpoint)

In [ ]:
(net.module if hasattr(net, "module") else net).load_state_dict(torch.load(f"{OUT}/best.pt")["model"])
e, er = evaluate(net, va)
print(f"{'session':<28}{'frames':>7}{'joints mm':>11}{'wrist-rel mm':>14}")
for s in dict.fromkeys(S_[va]):
    m = S_[va] == s
    print(f"{s:<28}{m.sum():>7}{e[m].mean():>11.1f}{er[m].mean():>14.1f}")
print(f"{'all':<28}{len(va):>7}{e.mean():>11.1f}{er.mean():>14.1f}   (mean-hand floor {base:.1f} mm)")
with torch.no_grad():
    net.eval()
    P = torch.cat([predict(net, Xg[va[b:b + 256]]) for b in range(0, len(va), 256)])
per_joint = (P - Yg[va]).norm(dim=-1).mean(0).cpu().numpy()
print("\nper joint, mm:", {n: round(float(v), 1) for n, v in zip(JOINTS, per_joint)})